# vLLM capture proof — P1 (TP=1, OLMoE)

**Question:** can we pull the three router streams (topk / logits / router-input, plan §1.6) out
of a vLLM forward pass and write them in the frozen trace format, with **topk.bin matching vLLM's
own selection**?

This is the EASY path: OLMoE fits one T4, so `tensor_parallel_size=1` and Python hooks reach the
router. The big TP=2 models are a later step.

**Kaggle Settings:** Accelerator **GPU T4 ×2** (only one card is used), Internet **On**, no dataset
needed.

**Run top-to-bottom ONCE, do NOT restart the kernel** (a restart reverts the pip installs to the
base image). Cell 1 audits via subprocess without importing torch; Cell 2 installs; Cell 3 clones
the repo for `src/`; Cell 4 loads OLMoE and PRINTS the router module tree; Cell 5 captures a few
docs and runs the gate.

**Paste back Cell 4's module-tree print and Cell 5's `MATCH:`/`SIZES:` lines** (or any traceback).


In [ ]:
# ============================================================================
# Cell 1 — runtime audit (subprocess; does NOT import torch into the kernel).
# ============================================================================
import subprocess, sys
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,compute_cap",
     "--format=csv"], capture_output=True, text=True).stdout, flush=True)
print("EXPECT two rows, compute_cap 7.5. Only card 0 is used (TP=1).")


In [ ]:
# ============================================================================
# Cell 2 — install vLLM + transformers (P0's confirmed recipe). Do NOT restart after.
# ============================================================================
VLLM_VERSION = "0.10.2"   # P0-confirmed: loads INT4 MoE on T4, moe_wna16 loader fixed.
import subprocess, sys


def sh(args):
    print("$", " ".join(args), flush=True)
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print((p.stdout or "")[-3000:], flush=True)
    print("exit:", p.returncode, flush=True)
    return p.returncode


# Purge first (any preinstalled version), then let vLLM pull its own matching torch.
sh([sys.executable, "-m", "pip", "uninstall", "-y",
    "vllm", "torch", "torchvision", "torchaudio", "transformers"])
sh([sys.executable, "-m", "pip", "install", "-q",
    f"vllm=={VLLM_VERSION}", "transformers==4.55.2"])
sh([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"])

print("\n--- verifying import in a clean subprocess ---", flush=True)
chk = subprocess.run(
    [sys.executable, "-c",
     "import torch, vllm; print('torch', torch.__version__); "
     "print('vllm', vllm.__version__); from vllm import LLM; print('IMPORT_OK')"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(chk.stdout, flush=True)
print(">>> Proceed to Cell 3 only if IMPORT_OK printed. DO NOT restart the kernel.")


In [ ]:
# ============================================================================
# Cell 3 — get src/ onto the path (the harness lives in the repo, not the notebook).
# ============================================================================
# The pipeline convention is: no logic in notebooks. src/capture/vllm_trace.py and
# src/traces/format.py do the work; this cell just makes them importable. Adjust REPO to wherever
# the repo is available in your Kaggle session (a private GitHub clone, or an attached dataset).
import os, sys, subprocess

REPO = "/kaggle/working/moe"   # EDIT if your repo is mounted elsewhere (e.g. a dataset path)
if not os.path.isdir(REPO):
    # If you attach the repo as a Kaggle dataset instead, set REPO to that path and skip the clone.
    print("Repo not found at", REPO, "-- set REPO to your clone/dataset path.")
else:
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    import src.capture.vllm_trace as _vt  # noqa: F401
    import src.traces.format as _fmt       # noqa: F401
    print("src importable from", REPO)


In [ ]:
# ============================================================================
# Cell 4 — load OLMoE at TP=1 and PRINT the router module tree (the T1.4 analogue).
# ============================================================================
import os
os.environ["VLLM_USE_V1"] = "0"                    # Turing needs the V0 engine (P0-confirmed).
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

MODEL_ID = "allenai/OLMoE-1B-7B-0125"              # ~4.2 GB, fp16, fits ONE T4. Not quantized.
N_MOE_LAYERS = 16                                   # models.yaml: olmoe-0125

import torch
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=1,                         # single card -> hooks reach the router
    enforce_eager=True,                             # required on Turing; also needed for hooks
    dtype="float16",
    max_model_len=2048,
    gpu_memory_utilization=0.85,
)

# Reach the underlying torch nn.Module. On the V0 engine this is the model runner's `model`.
runner = llm.llm_engine.model_executor.driver_worker.model_runner
model = runner.model
print("top-level model type:", type(model).__name__)

# Find the router (gate) module for each MoE layer. models.yaml predicts model.layers.{i}.mlp.gate
# for OLMoE, but vLLM may name/wrap it differently -- so DISCOVER it and print what we found.
named = dict(model.named_modules())
import re
def layer_of(name):
    m = re.search(r"layers\.(\d+)\.", name)
    return int(m.group(1)) if m else -1

# The router (raw logits) is the ReplicatedLinear `.gate`; the selection happens inside the
# FusedMoE `.experts`. models.yaml predicts model.layers.{i}.mlp.gate for OLMoE -- DISCOVER both
# and PRINT them so we confirm the vLLM names on the box before trusting any captured bytes.
gates = sorted([n for n in named if n.endswith(".mlp.gate") and layer_of(n) >= 0], key=layer_of)
experts = sorted([n for n in named if n.endswith(".mlp.experts") and layer_of(n) >= 0],
                 key=layer_of)
print(f"\nfound {len(gates)} .mlp.gate and {len(experts)} .mlp.experts modules")
for label, names in (("gate", gates[:3]), ("experts", experts[:3])):
    for n in names:
        print(f"   {label:8s} {n} -> {type(named[n]).__name__}")

assert len(gates) == N_MOE_LAYERS, f"expected {N_MOE_LAYERS} gates, found {len(gates)}: {gates}"
assert len(experts) == N_MOE_LAYERS, f"expected {N_MOE_LAYERS} experts, found {len(experts)}"
ROUTER_MODULES = [named[n] for n in gates]
EXPERTS_MODULES = [named[n] for n in experts]
# Sanity: the experts module should expose select_experts (vLLM's kernel-side selection).
have_sel = hasattr(EXPERTS_MODULES[0], "select_experts")
print(f"\nexperts[0] has select_experts: {have_sel}  ({type(EXPERTS_MODULES[0]).__name__})")
print("If False, paste the experts module type -- the topk capture point differs by vLLM version.")


In [ ]:
# ============================================================================
# Cell 5 — capture a few docs, run the faithfulness gate, check byte sizes.
# ============================================================================
# This exercises the WHOLE harness on real vLLM tensors. The selection gate (recomputed top-k ==
# vLLM's own) is the thing that proves capture is faithful; a mismatch raises SelectionMismatch.
import numpy as np, torch
from src.capture.vllm_trace import DocumentTrace, RouterCapture, gating_from_config
from src.traces.format import TraceSpec, expected_file_sizes

# OLMoE-0125 card (models.yaml). logit_tensor_used=ffn_moe_probs -> softmax; no router bias.
SPEC = TraceSpec(n_moe_layers=N_MOE_LAYERS, n_experts=64, top_k=8, hidden_dim=2048)
GATING = gating_from_config({"logit_tensor_used": "ffn_moe_probs", "has_router_bias": False})

PROMPTS = [
    "The quick brown fox jumps over the lazy dog.",
    "In 1969, humans first walked on the surface of the Moon.",
    "Photosynthesis converts light energy into chemical energy in plants.",
]
tok = llm.get_tokenizer()

# Capture BOTH the router (logits + input) AND vLLM's own select_experts topk_ids. The topk_ids
# come from vLLM's kernel path, INDEPENDENTLY of our recomputation -- that independence is what
# makes the gate inside DocumentTrace.put_layer a real faithfulness test (the T1.4 analogue), not
# a tautology. If they disagree on any token, put_layer raises SelectionMismatch.
cap = RouterCapture(ROUTER_MODULES, experts_modules=EXPERTS_MODULES)
cap.register()
matches = 0
try:
    for doc_id, prompt in enumerate(PROMPTS):
        cap.reset()
        ids = tok(prompt, add_special_tokens=True)["input_ids"]
        n = len(ids)
        # Prefill only, greedy; we read the hooks, not the generated text.
        llm.generate([{"prompt_token_ids": ids}],
                     SamplingParams(max_tokens=1, temperature=0.0))

        for stream, got in (("gate output", cap.outputs), ("gate input", cap.inputs),
                            ("select_experts topk", cap.topk_ids)):
            if sorted(got) != list(range(N_MOE_LAYERS)):
                raise RuntimeError(
                    f"{stream} fired for layers {sorted(got)}, expected 0..{N_MOE_LAYERS-1}")

        doc = DocumentTrace(SPEC, n_tokens=n,
                            capture_mask=[True] * n,   # capture all tokens in this tiny probe
                            gating=GATING)
        for L in range(N_MOE_LAYERS):
            # prefill packs this prompt's tokens contiguously; take the last n rows of each stream.
            logits = cap.outputs[L][-n:]
            router_input = cap.inputs[L][-n:]
            vllm_topk = cap.topk_ids[L][-n:]           # vLLM's OWN selection -> the gate's other side
            doc.put_layer(L, logits=logits, vllm_topk=vllm_topk, router_input=router_input)
        bufs = doc.to_buffers(token_ids=ids, doc_id=doc_id, global_token_base=doc_id * 4096)
        want = expected_file_sizes(SPEC, n, doc.n_captured)
        for name, blob in bufs.items():
            assert len(blob) == want[name], f"{name}: {len(blob)} != {want[name]}"
        matches += 1
        print(f"doc {doc_id}: {n} tokens, selection gate PASSED (recomputed == vLLM), sizes exact")
finally:
    cap.remove()

print(f"\nMATCH: {matches}/{len(PROMPTS)} documents -- recomputed top-8 == vLLM select_experts")
print("SIZES: OK" if matches == len(PROMPTS) else "SIZES: FAILED")
print("\nP1 (TP=1) PROVEN: three streams captured, topk.bin matches vLLM's kernel selection, bytes")
print("match the frozen format. Next: the TP=2 in-worker capturer for Qwen3-30B / Gemma-4.")
